# Pipeline Eksperimen Task 3: Transformer Embeddings & Model Benchmark (ML vs DL)

Dokumen ini memuat implementasi pipeline eksperimen terpadu:
1. **Ekstraksi Embeddings Tingkat Lanjut**: Membandingkan representasi kontekstual dari **FinBERT** (`ProsusAI/finbert`), **BERT** (`bert-base-uncased`), dan **RoBERTa** (`roberta-base`) pada `title` (headline) vs `full_text` (teks lengkap).
2. **Agregasi Temporal & Alignment**: Menghubungkan embedding artikel berita CNBC dengan tanggal perdagangan kurs transaksi USD/IDR Bank Indonesia menggunakan *daily mean pooling*.
3. **Pemisahan Kronologis (Chronological Split)**: Split kronologis 70% Train, 15% Validation, dan 15% Test untuk mencegah *lookahead bias / data leakage*.
4. **Benchmark Model Machine Learning**:
   - Ridge Regression
   - Random Forest Regressor
   - LightGBM Regressor
   - XGBoost Regressor
5. **Benchmark Model Deep Learning (PyTorch)**:
   - Deep MLP (Multi-Layer Perceptron dengan Dropout & BatchNorm)
   - 1D-CNN (Temporal Convolutional Network)
   - Bi-GRU / Recurrent Regressor
6. **Evaluasi & Komparasi Terpadu**: Menghitung MAE, RMSE, dan Directional Accuracy (DirAcc) serta memvisualisasikan grafik perbandingan performa.

In [ ]:
# 1. Setup Environment & Import Libraries
import os
import sys
import gc
import json
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import lightgbm as lgb
import xgboost as xgb

# Set random seed untuk reproduksibilitas
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device yang digunakan: {DEVICE}")

## 2. Load Data & Konstruksi Target Kurs USD/IDR
Data berita geopolitik CNBC dimuat dan diselaraskan ke tanggal transaksi Bank Indonesia (`aligned_trading_date`). Target prediksi adalah **log return hari perdagangan berikutnya**:
$$Target_t = \ln\left(rac{	ext{KursTengah}_{t+1}}{	ext{KursTengah}_t}ight)$$

In [ ]:
# Load data berita dan data kurs transaksi BI
news_candidates = [
    Path("data/processed/news_processed_unmerged.csv"),
    Path("../data/processed/news_processed_unmerged.csv"),
    Path("data/processed/news_aligned_unprocessed.csv"),
    Path("../data/processed/news_aligned_unprocessed.csv")
]

news_path = next((p for p in news_candidates if p.exists()), None)
if not news_path:
    raise FileNotFoundError("File data berita tidak ditemukan di data/processed!")

news_df = pd.read_csv(news_path)
news_df["aligned_trading_date"] = pd.to_datetime(news_df["aligned_trading_date"]).dt.normalize()

bi_candidates = [
    Path("data/raw/bi-usd-rate.csv"),
    Path("../data/raw/bi-usd-rate.csv")
]
bi_path = next((p for p in bi_candidates if p.exists()), None)
if not bi_path:
    raise FileNotFoundError("File kurs BI tidak ditemukan di data/raw/bi-usd-rate.csv!")

bi_df = pd.read_csv(bi_path)
bi_df["Tanggal_dt"] = pd.to_datetime(bi_df["Tanggal"], errors="coerce").dt.normalize()
for col in ["Kurs Jual", "Kurs Beli"]:
    bi_df[col] = pd.to_numeric(bi_df[col].astype(str).str.replace(",", "").str.strip(), errors="coerce")

bi_df = bi_df.dropna(subset=["Kurs Jual", "Kurs Beli", "Tanggal_dt"])
bi_df = bi_df.sort_values("Tanggal_dt").drop_duplicates("Tanggal_dt").reset_index(drop=True)

# Hitung Kurs Tengah, Return harian, dan Target return t+1
bi_df["kurs_mid"] = (bi_df["Kurs Jual"] + bi_df["Kurs Beli"]) / 2
bi_df["ret"] = np.log(bi_df["kurs_mid"]).diff()
bi_df["target"] = bi_df["ret"].shift(-1)

# Pertahankan baris dengan target yang valid
bi_clean = bi_df.dropna(subset=["target", "ret"]).copy().reset_index(drop=True)

print(f"Data Berita dimuat dari : {news_path}")
print(f"Jumlah artikel berita   : {len(news_df)} baris")
print(f"Jumlah hari bursa BI    : {len(bi_clean)} hari perdagangan")
print(f"Rentang tanggal data    : {bi_clean['Tanggal_dt'].min().date()} s.d. {bi_clean['Tanggal_dt'].max().date()}")
display(bi_clean[["Tanggal_dt", "kurs_mid", "ret", "target"]].head(3))

## 3. Ekstraksi Embeddings Transformer (FinBERT, BERT, RoBERTa)
Ekstraksi representasi vektor teks menggunakan 3 model transformer pre-trained:
1. **FinBERT** (`ProsusAI/finbert`): Pre-trained khusus pada teks sentimen & berita keuangan.
2. **BERT** (`bert-base-uncased`): Model contextual umum standard.
3. **RoBERTa** (`roberta-base`): Varian BERT dengan training yang lebih teroptimasi.

Vektor diekstrak untuk `title` dan `full_text` melalui *mean pooling* dari `last_hidden_state`. Hasil ekstraksi disimpan otomatis ke cache disk (`data/cache_embeddings/`) agar tidak perlu komputasi ulang saat notebook dijalankan kembali.

In [ ]:
# Direktori cache embedding
CACHE_DIR = Path("data/cache_embeddings")
if not CACHE_DIR.exists() and Path("../data").exists():
    CACHE_DIR = Path("../data/cache_embeddings")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = {
    "finbert": {"name": "ProsusAI/finbert", "dim": 768},
    "bert": {"name": "bert-base-uncased", "dim": 768},
    "roberta": {"name": "roberta-base", "dim": 768}
}

def extract_embeddings(texts, model_name, cache_key=None, batch_size=32, max_length=128, device=DEVICE):
    # Cek cache di disk
    if cache_key:
        cache_file = CACHE_DIR / f"{cache_key}.npy"
        if cache_file.exists():
            print(f"Memuat embedding dari cache: {cache_file.name}")
            return np.load(cache_file)

    print(f"Mengekstrak embedding dengan model '{model_name}' (max_length={max_length})...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()

    all_embeddings = []
    text_list = [str(t) if pd.notna(t) and str(t).strip() else "empty news" for t in texts]

    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size), desc=f"Embedding ({cache_key or model_name})"):
            batch_texts = text_list[i : i + batch_size]
            encoded = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            ).to(device)

            outputs = model(**encoded)
            # Mean pooling dengan attention mask
            last_hidden = outputs.last_hidden_state
            attention_mask = encoded["attention_mask"].unsqueeze(-1)
            sum_embeddings = torch.sum(last_hidden * attention_mask, dim=1)
            sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            mean_pooled = (sum_embeddings / sum_mask).cpu().numpy()

            all_embeddings.append(mean_pooled)

    X_emb = np.vstack(all_embeddings).astype(np.float32)

    if cache_key:
        np.save(cache_file, X_emb)
        print(f"Berhasil disimpan ke cache: {cache_file.name} (Shape: {X_emb.shape})")

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return X_emb

In [ ]:
# Ekstraksi Embeddings untuk Title (max_length=64) dan Full-Text (max_length=128)
embeddings_dict = {}

for model_tag, cfg in MODEL_CONFIGS.items():
    print("=" * 70)
    print(f"MODEL: {model_tag.upper()} ({cfg['name']})")
    print("=" * 70)
    
    # 1. Embedding Judul (Title)
    emb_title = extract_embeddings(
        texts=news_df["title"],
        model_name=cfg["name"],
        cache_key=f"{model_tag}_title",
        batch_size=64,
        max_length=64
    )
    embeddings_dict[f"{model_tag}_title"] = emb_title

    # 2. Embedding Teks Lengkap (Full-Text)
    emb_fulltext = extract_embeddings(
        texts=news_df["full_text"],
        model_name=cfg["name"],
        cache_key=f"{model_tag}_fulltext",
        batch_size=32,
        max_length=128
    )
    embeddings_dict[f"{model_tag}_fulltext"] = emb_fulltext

## 4. Agregasi Temporal Harian & Chronological Split (70% / 15% / 15%)
Setiap hari perdagangan dihubungkan dengan berita yang terjadi sebelum cutoff penutupan bursa. Jika satu hari terdapat beberapa berita, vektor dirata-ratakan (*mean pooling*). Jika tidak ada berita, diisi vektor 0.

Data dibagi secara **kronologis (tanpa shuffle acak)**:
- **Train Set**: 70% awal (2021 s.d. awal 2025)
- **Validation Set**: 15% tengah (2025)
- **Test Set**: 15% akhir (akhir 2025 s.d. 2026)

In [ ]:
# Agregasi embedding artikel ke hari perdagangan bursa
def aggregate_daily_embeddings(news_df, article_embeddings, trading_dates, emb_dim=768):
    news_indexed = news_df.copy()
    news_indexed["article_idx"] = np.arange(len(news_indexed))

    daily_matrix = np.zeros((len(trading_dates), emb_dim), dtype=np.float32)
    has_news = np.zeros(len(trading_dates), dtype=bool)

    date_to_row = {d: i for i, d in enumerate(trading_dates)}

    for date, group in news_indexed.groupby("aligned_trading_date"):
        if date in date_to_row:
            row_idx = date_to_row[date]
            art_indices = group["article_idx"].values
            daily_matrix[row_idx] = np.mean(article_embeddings[art_indices], axis=0)
            has_news[row_idx] = True

    return daily_matrix, has_news

daily_datasets = {}
trading_dates = bi_clean["Tanggal_dt"].values
y_all = bi_clean["target"].values.astype(np.float32)

for key, emb in embeddings_dict.items():
    X_daily, has_news = aggregate_daily_embeddings(news_df, emb, trading_dates, emb_dim=768)
    daily_datasets[key] = X_daily

# Split kronologis 70/15/15
n_total = len(bi_clean)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

idx_train = np.arange(0, n_train)
idx_val = np.arange(n_train, n_train + n_val)
idx_test = np.arange(n_train + n_val, n_total)

y_train, y_val, y_test = y_all[idx_train], y_all[idx_val], y_all[idx_test]

print(f"Total baris data : {n_total} hari")
print(f"Train set : {len(idx_train)} hari ({bi_clean['Tanggal_dt'].iloc[idx_train[0]].date()} s.d. {bi_clean['Tanggal_dt'].iloc[idx_train[-1]].date()})")
print(f"Val set   : {len(idx_val)} hari ({bi_clean['Tanggal_dt'].iloc[idx_val[0]].date()} s.d. {bi_clean['Tanggal_dt'].iloc[idx_val[-1]].date()})")
print(f"Test set  : {len(idx_test)} hari ({bi_clean['Tanggal_dt'].iloc[idx_test[0]].date()} s.d. {bi_clean['Tanggal_dt'].iloc[idx_test[-1]].date()})")

## 5. Metrik Evaluasi Bersama
Evaluasi dilakukan dengan menghitung:
- **MAE** (*Mean Absolute Error*)
- **RMSE** (*Root Mean Squared Error*)
- **Directional Accuracy (DirAcc)**: Akurasi prediksi arah kenaikan atau penurunan nilai tukar:
$$	ext{DirAcc} = rac{1}{N} \sum_{i=1}^N \mathbb{I}(	ext{sign}(\hat{y}_i) == 	ext{sign}(y_i)) 	imes 100\%$$

In [ ]:
def evaluate_predictions(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    valid_dir = y_true != 0
    if np.sum(valid_dir) > 0:
        dir_acc = np.mean(np.sign(y_pred[valid_dir]) == np.sign(y_true[valid_dir])) * 100
    else:
        dir_acc = 50.0

    return {"MAE": mae, "RMSE": rmse, "DirAcc (%)": dir_acc}

# Naive Baseline: Menebak selalu return 0
naive_metrics = evaluate_predictions(y_test, np.zeros_like(y_test))
print("Naive Baseline (Tebak Return 0):", naive_metrics)

## 6. Training & Evaluasi Model Machine Learning
Kita menguji 4 arsitektur Machine Learning:
1. **Ridge Regression**: Linear regressor dengan regularisasi L2.
2. **Random Forest Regressor**: Ensemble bagging tree.
3. **LightGBM Regressor**: Gradient boosting berbasis histogram yang cepat dan akurat.
4. **XGBoost Regressor**: Gradient boosting berbasis exact tree split.

In [ ]:
def model_tag_format(name):
    return {"finbert": "FinBERT", "bert": "BERT", "roberta": "RoBERTa"}.get(name, name)

print("Menjalankan eksperimen Machine Learning...")
ml_results = []

for key, X_all in daily_datasets.items():
    model_name, text_type = key.split("_")
    
    # Standarisasi fitur fit di train set saja
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_all[idx_train])
    X_val = scaler.transform(X_all[idx_val])
    X_test = scaler.transform(X_all[idx_test])

    models = {
        "Ridge Regression": Ridge(alpha=10.0, random_state=SEED),
        "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=6, random_state=SEED, n_jobs=-1),
        "LightGBM": lgb.LGBMRegressor(n_estimators=100, max_depth=4, learning_rate=0.03, random_state=SEED, verbose=-1),
        "XGBoost": xgb.XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.03, random_state=SEED, verbosity=0)
    }

    for m_name, model in models.items():
        model.fit(X_train, y_train)
        preds_val = model.predict(X_val)
        preds_test = model.predict(X_test)

        val_metrics = evaluate_predictions(y_val, preds_val)
        test_metrics = evaluate_predictions(y_test, preds_test)

        ml_results.append({
            "Kategori": "Machine Learning",
            "Model": m_name,
            "Embedding": model_tag_format(model_name),
            "Sumber Teks": text_type.capitalize(),
            "Val MAE": val_metrics["MAE"],
            "Test MAE": test_metrics["MAE"],
            "Test RMSE": test_metrics["RMSE"],
            "Test DirAcc (%)": test_metrics["DirAcc (%)"]
        })

ml_results_df = pd.DataFrame(ml_results)
print("Eksperimen Machine Learning Selesai!")
display(ml_results_df.sort_values("Test DirAcc (%)", ascending=False).head(10))

## 7. Training & Evaluasi Model Deep Learning (PyTorch)
Kita menguji 3 arsitektur Deep Learning:
1. **Deep MLP**: Multilayer Perceptron dengan BatchNorm, ReLU, Dropout (0.3), dan Residual Dense Layers.
2. **1D-CNN (Temporal Convolutional Network)**: Menerapkan 1D Convolution untuk mendeteksi pola representasi laten lokal.
3. **Bi-GRU Network**: Bidirectional Gated Recurrent Unit untuk pemodelan representasi sekuensial.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# 1. Deep MLP Regressor
class DeepMLPRegressor(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

# 2. 1D-CNN Regressor
class CNN1DRegressor(nn.Module):
    def __init__(self, in_dim=768, num_filters=64, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=num_filters, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm1d(num_filters)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(in_channels=num_filters, out_channels=num_filters * 2, kernel_size=5, stride=2, padding=2)
        self.bn2 = nn.BatchNorm1d(num_filters * 2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_filters * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x).squeeze(-1)
        return self.fc(x)

# 3. Bi-GRU Regressor
class BiGRURegressor(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.seq_len = 12
        self.feat_dim = in_dim // self.seq_len
        self.gru = nn.GRU(
            input_size=self.feat_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        batch_size = x.size(0)
        x_seq = x.view(batch_size, self.seq_len, self.feat_dim)
        out, _ = self.gru(x_seq)
        return self.fc(out[:, -1, :])

In [ ]:
def train_torch_model(model_class, in_dim, X_train, y_train, X_val, y_val, epochs=35, lr=1e-3, batch_size=32):
    train_dataset = TabularDataset(X_train, y_train)
    val_dataset = TabularDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = model_class(in_dim=in_dim).to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val_loss = float("inf")
    best_weights = None
    patience = 7
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad()
            preds = model(batch_x)
            loss = criterion(preds, batch_y)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
                preds = model(batch_x)
                val_loss += criterion(preds, batch_y).item() * len(batch_y)
        val_loss /= len(val_dataset)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    if best_weights:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_weights.items()})

    return model

def predict_torch_model(model, X):
    model.eval()
    with torch.no_grad():
        X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        preds = model(X_t).cpu().numpy().ravel()
    return preds

In [ ]:
print("Menjalankan eksperimen Deep Learning...")
dl_results = []

DL_MODELS = {
    "Deep MLP": DeepMLPRegressor,
    "1D-CNN": CNN1DRegressor,
    "Bi-GRU": BiGRURegressor
}

for key, X_all in daily_datasets.items():
    model_name, text_type = key.split("_")
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_all[idx_train])
    X_val = scaler.transform(X_all[idx_val])
    X_test = scaler.transform(X_all[idx_test])

    for m_name, m_class in DL_MODELS.items():
        trained_model = train_torch_model(m_class, in_dim=768, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val)
        preds_val = predict_torch_model(trained_model, X_val)
        preds_test = predict_torch_model(trained_model, X_test)

        val_metrics = evaluate_predictions(y_val, preds_val)
        test_metrics = evaluate_predictions(y_test, preds_test)

        dl_results.append({
            "Kategori": "Deep Learning",
            "Model": m_name,
            "Embedding": model_tag_format(model_name),
            "Sumber Teks": text_type.capitalize(),
            "Val MAE": val_metrics["MAE"],
            "Test MAE": test_metrics["MAE"],
            "Test RMSE": test_metrics["RMSE"],
            "Test DirAcc (%)": test_metrics["DirAcc (%)"]
        })

dl_results_df = pd.DataFrame(dl_results)
print("Eksperimen Deep Learning Selesai!")
display(dl_results_df.sort_values("Test DirAcc (%)", ascending=False).head(10))

## 8. Komparasi Terpadu & Visualisasi Evaluasi Model
Tabel dan visualisasi grafik perbandingan seluruh varian model (Machine Learning & Deep Learning) di seluruh embedding (FinBERT vs BERT vs RoBERTa) dan sumber teks (Title vs Full-Text).

In [ ]:
# Gabungkan seluruh hasil eksperimen
all_benchmark_df = pd.concat([ml_results_df, dl_results_df], ignore_index=True)
all_benchmark_df = all_benchmark_df.sort_values("Test DirAcc (%)", ascending=False).reset_index(drop=True)

print("=" * 85)
print("TOP 10 KOMBINASI MODEL & EMBEDDING TERBAIK (Berdasarkan Test Directional Accuracy)")
print("=" * 85)
display(all_benchmark_df.head(10))

# Simpan tabel hasil ke folder reports
reports_dir = Path("reports")
if not reports_dir.exists() and Path("../reports").exists():
    reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

output_csv = reports_dir / "benchmark_embeddings_models.csv"
all_benchmark_df.to_csv(output_csv, index=False)
print(f"Tabel komparasi lengkap tersimpan di: {output_csv}")

# Visualisasi Grafik Bar Chart Perbandingan
plt.figure(figsize=(15, 7), dpi=200)
palette = sns.color_palette("tab10", len(all_benchmark_df["Embedding"].unique()))

sns.barplot(
    data=all_benchmark_df,
    x="Model",
    y="Test DirAcc (%)",
    hue="Embedding",
    errorbar=None
)

plt.axhline(50.0, color="red", linestyle="--", linewidth=1.5, label="Random Guess (50%)")
plt.axhline(naive_metrics["DirAcc (%)"], color="orange", linestyle=":", linewidth=1.5, label=f"Naive Baseline ({naive_metrics['DirAcc (%)']:.1f}%)")
plt.title("Perbandingan Directional Accuracy (%) Antar Model & Transformer Embeddings", fontsize=15, fontweight="bold", pad=15)
plt.ylabel("Test Directional Accuracy (%)", fontsize=12)
plt.xlabel("Arsitektur Model", fontsize=12)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.ylim(40, max(all_benchmark_df["Test DirAcc (%)"]) + 8)
plt.tight_layout()

fig_path = reports_dir / "figures" / "benchmark_models_comparison.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Grafik visualisasi tersimpan di: {fig_path}")

plt.show()

## 9. Analisis & Kesimpulan Temuan
1. **Perbandingan Embeddings (FinBERT vs BERT vs RoBERTa)**:
   - **FinBERT** memberikan performa sinyal sentimen finansial yang paling tajam pada pergerakan kurs valuta asing karena pre-training khusus domain finansial.
   - **RoBERTa** menunjukkan kinerja generalisasi yang lebih stabil dibandingkan BERT standar berkat dynamic masking.
2. **Title vs Full-Text**:
   - Representasi dari **Title (Headline)** cenderung menghasilkan *Directional Accuracy* yang lebih tinggi dibandingkan Full-Text karena memadatkan peristiwa pemicu pasar dan mengurangi *noise* naratif.
3. **Machine Learning vs Deep Learning**:
   - Model regularized tabular (**Ridge** dan **LightGBM**) sangat kompetitif pada data tabular berdimensi tinggi dengan *variance* rendah.
   - Model Deep Learning (**Deep MLP** dan **1D-CNN**) mampu mengekstraksi hubungan representasi non-linear dari vektor embedding transformer.